# 02 — Zero-shot NLI baseline: demo with task-type-aware chunking (Step 2.4)

Demonstrates the NLI baseline end-to-end on the validation split using the corrected
API (ADR-008): the caller first turns a raw `source_info` into evidence chunks with
`chunk_context(task_type, source_info)`, then passes those chunks to
`detect(context_chunks, response, ...)` / `score_response(context_chunks, response)`.

**Data note.** The processed parquets store head-truncated token IDs (ADR-004), not
raw text, and the NLI baseline needs the full untruncated `source_info`. So we take the
val split's `source_id`s from `response_level_val.parquet` and pull full-text rows from
the raw merged dataset.

Heavy wall-clock timing (30-row sample, per-task extrapolation) lives in
`scripts/time_baseline_sample.py`, which streams progress to `results/timing_log.txt`.

In [ ]:
# Run from the repo root so `src` imports and repo-root-relative data paths resolve,
# whether this notebook is executed from notebooks/ or the project root.
import os

if os.path.basename(os.getcwd()) == "notebooks":
    os.chdir("..")
print("cwd:", os.getcwd())

In [ ]:
import pandas as pd

from src.data.preprocess import load_merged_dataframe
from src.data.context_chunking import chunk_context
from src.models.nli_baseline import NLIHallucinationDetector, response_color

VAL_PARQUET = "data/processed/response_level_val.parquet"

In [ ]:
# Val-split membership from the parquet; full text (incl. raw source_info) from the dataset.
val_meta = pd.read_parquet(VAL_PARQUET, columns=["source_id"])
val_source_ids = set(val_meta["source_id"].unique())

merged = load_merged_dataframe()
val_text = merged[merged["source_id"].isin(val_source_ids)][
    ["source_id", "task_type", "source_info", "response", "labels"]
].copy()
val_text["label_response"] = val_text["labels"].apply(lambda labels: int(len(labels) > 0))
print(f"val parquet rows: {len(val_meta)} | reconstructed full-text val rows: {len(val_text)}")

In [ ]:
detector = NLIHallucinationDetector.from_pretrained()
print("device:", detector.device)

## One worked example per task_type

`chunk_context` -> `detect`, showing the traffic-light color and the first few sentence verdicts.

In [ ]:
for task_type in ["Summary", "QA", "Data2txt"]:
    row = val_text[val_text["task_type"] == task_type].iloc[0]
    context_chunks = chunk_context(row["task_type"], row["source_info"])
    result = detector.detect(context_chunks, row["response"])

    print(f"\n=== {task_type} (source_id={row['source_id']}) ===")
    print(f"context chunks: {len(context_chunks)} | response sentences: {len(result.verdicts)}")
    print(f"{response_color(result)}  response_hallucinated={result.response_hallucinated}")
    for verdict in result.verdicts[:3]:
        print(
            f"  [{verdict.flag:12s}] ent={verdict.entailment:.2f} con={verdict.contradiction:.2f}"
            f"  :: {verdict.sentence[:80]}"
        )